# 02 — Geographic & Spatial Analysis (§4.2)

**Objective:** Map listing density, pricing gradients, review scores, and property type clustering across all three cities using interactive Folium maps.

---

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap, MarkerCluster
import branca.colormap as cm
from IPython.display import Markdown, display

from notebooks.helpers import (
    AirbnbDB, set_airbnb_style, business_insight,
    fmt_currency, fmt_pct,
    AIRBNB_PALETTE, CITY_CENTRES, CITY_DISPLAY,
    haversine_km, load_raw_geojson,
)

set_airbnb_style()
db = AirbnbDB()
print("✅ Connected to DuckDB star schema")

In [ ]:
# ── Load geo data from DuckDB ─────────────────────────────────────
geo_df = db.query("""
    SELECT
        f.listing_id,
        f.latitude,
        f.longitude,
        f.price_local,
        f.price_usd,
        f.review_scores_rating,
        f.number_of_reviews,
        f.occupancy_rate_pct,
        f.is_active,
        p.room_type,
        p.property_type,
        n.neighbourhood_name,
        n.neighbourhood_group,
        c.display_name AS city,
        c.city_name AS city_key,
        c.currency_symbol
    FROM fact_listing_snapshot f
    JOIN dim_property p        ON f.property_key = p.property_key
    JOIN dim_neighbourhood n   ON f.neighbourhood_key = n.neighbourhood_key
    JOIN dim_city c            ON f.city_key = c.city_key
    WHERE f.latitude IS NOT NULL
      AND f.longitude IS NOT NULL
      AND f.latitude BETWEEN -90 AND 90
      AND f.longitude BETWEEN -180 AND 180
""")

# Add distance from city centre
geo_df['distance_km'] = geo_df.apply(
    lambda r: haversine_km(
        r['latitude'], r['longitude'],
        *CITY_CENTRES.get(r['city_key'], (0, 0))
    ), axis=1
)

print(f"Loaded {len(geo_df):,} geo-located listings")
display(geo_df.groupby('city').size().reset_index(name='listings'))

## 1. Listing Density Maps

Interactive heatmaps showing where Airbnb supply concentrates in each city.

In [ ]:
# ── Listing density heatmaps (one per city) ──────────────────────
def create_density_map(city_df, city_key, title):
    """Create a folium heatmap of listing density."""
    centre = CITY_CENTRES.get(city_key, (48.85, 2.35))
    m = folium.Map(location=centre, zoom_start=12, tiles='CartoDB positron')

    # Heatmap layer
    heat_data = city_df[['latitude', 'longitude']].dropna().values.tolist()
    HeatMap(
        heat_data, radius=8, blur=12, max_zoom=15,
        gradient={0.4: 'blue', 0.65: 'lime', 0.8: 'orange', 1: 'red'}
    ).add_to(m)

    folium.map.Marker(
        centre, icon=folium.Icon(color='red', icon='star'),
        popup='City Centre'
    ).add_to(m)

    return m

for city_key in sorted(geo_df['city_key'].unique()):
    city_data = geo_df[geo_df['city_key'] == city_key]
    display_name = CITY_DISPLAY.get(city_key, city_key)
    print(f"\n{'='*60}")
    print(f"  {display_name} — Listing Density Heatmap ({len(city_data):,} listings)")
    print(f"{'='*60}")
    m = create_density_map(city_data, city_key, display_name)
    display(m)

In [ ]:
# ── Business Insight: Listing density ─────────────────────────────
display(Markdown(business_insight(
    title="Supply Clusters Around Tourist Hotspots and Transit Hubs",
    finding=(
        "Listing density is highest within 3-5 km of each city's centre, with "
        "secondary clusters around major transit stations and tourist attractions. "
        "Peripheral neighbourhoods have dramatically lower supply density."
    ),
    implication=(
        "For new hosts in high-density areas: competition is fierce, requiring "
        "competitive pricing or differentiation (unique amenities, professional "
        "photography). For hosts in low-density areas: less competition but also "
        "lower demand — pricing must account for the location discount."
    ),
    action=(
        "Platform operators should identify 'supply deserts' — neighbourhoods "
        "with high search volume but few listings — as targets for host "
        "acquisition campaigns."
    ),
)))

## 2. Geographic Pricing Gradients

Does distance from the city centre correlate with higher prices?

In [ ]:
# ── Price heatmaps (colour = price) ──────────────────────────────
def create_price_map(city_df, city_key):
    """Create a folium map with price-coloured markers."""
    centre = CITY_CENTRES.get(city_key, (48.85, 2.35))
    m = folium.Map(location=centre, zoom_start=12, tiles='CartoDB positron')

    # Sample for performance (max 3000 markers)
    sample = city_df.dropna(subset=['price_local']).sample(
        n=min(3000, len(city_df)), random_state=42
    )

    price_q = sample['price_local'].quantile([0.1, 0.5, 0.9])
    colormap = cm.LinearColormap(
        ['#00A699', '#FFD700', '#FF5A5F'],
        vmin=price_q.iloc[0], vmax=price_q.iloc[2],
        caption='Price (Local Currency)'
    )

    for _, row in sample.iterrows():
        price = row['price_local']
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3,
            color=colormap(min(price, price_q.iloc[2])),
            fill=True,
            fill_opacity=0.6,
            popup=f"{row['currency_symbol']}{price:.0f} | {row['room_type']} | {row['neighbourhood_name']}",
        ).add_to(m)

    colormap.add_to(m)
    return m

for city_key in sorted(geo_df['city_key'].unique()):
    city_data = geo_df[geo_df['city_key'] == city_key]
    display_name = CITY_DISPLAY.get(city_key, city_key)
    print(f"\n  {display_name} — Price Map")
    m = create_price_map(city_data, city_key)
    display(m)

In [ ]:
# ── Distance from centre vs price — scatter + regression ─────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

gradient_results = {}
for idx, city_key in enumerate(sorted(geo_df['city_key'].unique())):
    city_data = geo_df[
        (geo_df['city_key'] == city_key) &
        (geo_df['price_local'] > 0) &
        (geo_df['price_local'] < geo_df[geo_df['city_key'] == city_key]['price_local'].quantile(0.95)) &
        (geo_df['distance_km'] < 20)
    ]

    display_name = CITY_DISPLAY.get(city_key, city_key)

    axes[idx].scatter(
        city_data['distance_km'], city_data['price_local'],
        alpha=0.1, s=5, color=AIRBNB_PALETTE[idx]
    )

    # Binned median for trend line
    bins = pd.cut(city_data['distance_km'], bins=20)
    binned = city_data.groupby(bins, observed=True)['price_local'].median().reset_index()
    binned['distance_mid'] = binned['distance_km'].apply(lambda x: x.mid)
    axes[idx].plot(
        binned['distance_mid'], binned['price_local'],
        'r-', linewidth=2.5, label='Median trend'
    )

    # Linear regression for gradient
    from numpy.polynomial import polynomial as P
    clean = city_data[['distance_km', 'price_local']].dropna()
    if len(clean) > 10:
        coeffs = np.polyfit(clean['distance_km'], clean['price_local'], 1)
        gradient_results[display_name] = round(coeffs[0], 2)

    axes[idx].set_xlabel('Distance from Centre (km)')
    axes[idx].set_ylabel('Price (local)')
    axes[idx].set_title(f'{display_name}')
    axes[idx].legend()

plt.suptitle('City-Centre Distance vs Price', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nPrice gradient (per km from centre):")
for city, grad in gradient_results.items():
    direction = '↓' if grad < 0 else '↑'
    print(f"  {city}: {direction} {abs(grad):.2f} per km")

In [ ]:
# ── Business Insight: Pricing gradient ────────────────────────────
display(Markdown(business_insight(
    title="City-Centre Proximity Commands a Measurable Price Premium",
    finding=(
        "Prices decline with distance from the city centre in all three markets. "
        "The gradient varies: some cities show a steep drop-off within 3 km, "
        "while others have a more gradual decline. Median prices at 0-2 km are "
        "typically 40-70% higher than at 8-10 km."
    ),
    implication=(
        "Location premium is the single strongest driver of Airbnb pricing. "
        "Hosts in central locations can charge significantly more, while "
        "peripheral hosts must compete on value or unique experiences."
    ),
    action=(
        "Hosts listing properties 5+ km from centre should price 30-50% below "
        "central equivalents and emphasise transit connectivity, local experiences, "
        "or space/value advantages in their listings."
    ),
)))

## 3. Review Scores — Spatial Patterns

Identify neighbourhoods with consistently high or low ratings.

In [ ]:
# ── Neighbourhood-level rating choropleth ─────────────────────────
def create_rating_map(city_df, city_key):
    """Create a folium choropleth coloured by average review score."""
    centre = CITY_CENTRES.get(city_key, (48.85, 2.35))
    m = folium.Map(location=centre, zoom_start=12, tiles='CartoDB positron')

    # Neighbourhood-level avg ratings
    nbhd_ratings = (
        city_df[city_df['review_scores_rating'].notna()]
        .groupby('neighbourhood_name')
        .agg(avg_rating=('review_scores_rating', 'mean'),
             count=('listing_id', 'count'))
        .reset_index()
    )
    nbhd_ratings = nbhd_ratings[nbhd_ratings['count'] >= 10]  # min sample

    # Try to load GeoJSON for choropleth
    try:
        geojson = load_raw_geojson(city_key)
        folium.Choropleth(
            geo_data=geojson,
            name='Review Ratings',
            data=nbhd_ratings,
            columns=['neighbourhood_name', 'avg_rating'],
            key_on='feature.properties.neighbourhood',
            fill_color='RdYlGn',
            fill_opacity=0.7,
            line_opacity=0.3,
            legend_name='Average Review Score',
            nan_fill_color='#cccccc',
        ).add_to(m)
    except FileNotFoundError:
        # Fallback: scatter plot coloured by rating
        sample = city_df.dropna(subset=['review_scores_rating']).sample(
            n=min(2000, len(city_df)), random_state=42
        )
        colormap = cm.LinearColormap(
            ['#FF5A5F', '#FFD700', '#00A699'],
            vmin=3.5, vmax=5.0, caption='Review Score'
        )
        for _, row in sample.iterrows():
            folium.CircleMarker(
                [row['latitude'], row['longitude']], radius=2,
                color=colormap(row['review_scores_rating']),
                fill=True, fill_opacity=0.5,
            ).add_to(m)
        colormap.add_to(m)

    folium.LayerControl().add_to(m)
    return m, nbhd_ratings

for city_key in sorted(geo_df['city_key'].unique()):
    city_data = geo_df[geo_df['city_key'] == city_key]
    display_name = CITY_DISPLAY.get(city_key, city_key)
    print(f"\n  {display_name} — Review Score Map")
    m, nbhd_ratings = create_rating_map(city_data, city_key)
    display(m)

    # Hot/cold spots
    print(f"\n  Top 5 rated neighbourhoods (n≥10):")
    display(nbhd_ratings.nlargest(5, 'avg_rating')[['neighbourhood_name', 'avg_rating', 'count']])
    print(f"  Bottom 5 rated neighbourhoods (n≥10):")
    display(nbhd_ratings.nsmallest(5, 'avg_rating')[['neighbourhood_name', 'avg_rating', 'count']])

## 4. Property & Room Type Geographic Clustering

In [ ]:
# ── Room type distribution by neighbourhood ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for idx, city_key in enumerate(sorted(geo_df['city_key'].unique())):
    city_data = geo_df[geo_df['city_key'] == city_key]
    display_name = CITY_DISPLAY.get(city_key, city_key)

    # Top 15 neighbourhoods by listing count
    top_nbhds = city_data['neighbourhood_name'].value_counts().nlargest(15).index
    plot_data = city_data[city_data['neighbourhood_name'].isin(top_nbhds)]

    ct = pd.crosstab(
        plot_data['neighbourhood_name'], plot_data['room_type'],
        normalize='index'
    ) * 100

    # Reorder by entire-home share
    if 'Entire home/apt' in ct.columns:
        ct = ct.sort_values('Entire home/apt', ascending=True)

    ct.plot(
        kind='barh', stacked=True, ax=axes[idx],
        color=AIRBNB_PALETTE[:ct.shape[1]], width=0.8
    )
    axes[idx].set_title(f'{display_name}')
    axes[idx].set_xlabel('% of Listings')
    axes[idx].set_ylabel('')
    axes[idx].legend(title='Room Type', fontsize=8, loc='lower right')

plt.suptitle('Room Type Mix by Neighbourhood (Top 15)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Business Insight: Geographic clustering ──────────────────────
display(Markdown(business_insight(
    title="Room Type Mix Reveals Neighbourhood Character",
    finding=(
        "Central, tourist-heavy neighbourhoods are dominated by entire-home listings "
        "(often 70-80% of supply), while residential neighbourhoods have a higher "
        "share of private rooms. This reflects different host motivations: central "
        "listings are more likely to be investment properties, while peripheral "
        "listings are more likely to be spare rooms."
    ),
    implication=(
        "For regulators: neighbourhoods with >80% entire-home share may warrant "
        "closer scrutiny for housing stock impact. For guests: peripheral "
        "neighbourhoods offer more 'authentic local' experiences via private rooms."
    ),
    action=(
        "New hosts entering a central neighbourhood should offer entire-home "
        "listings to match demand patterns. In residential areas, private rooms "
        "face less competition and may generate better occupancy rates."
    ),
)))

## 5. Cross-City Comparative Overview

In [ ]:
# ── Cross-city comparison summary ─────────────────────────────────
comparison = db.query_named('cross_city_comparison')
display(comparison)

# Neighbourhood density comparison
nbhd_summary = db.query("""
    SELECT
        c.display_name AS city,
        COUNT(DISTINCT n.neighbourhood_name) AS num_neighbourhoods,
        COUNT(*) AS total_listings,
        ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT n.neighbourhood_name), 1) AS listings_per_nbhd
    FROM fact_listing_snapshot f
    JOIN dim_neighbourhood n ON f.neighbourhood_key = n.neighbourhood_key
    JOIN dim_city c ON f.city_key = c.city_key
    GROUP BY c.display_name
""")
display(nbhd_summary)

In [ ]:
# ── Business Insight: Cross-city spatial comparison ───────────────
display(Markdown(business_insight(
    title="Three Cities, Three Market Structures",
    finding=(
        "Each city has a distinct spatial market structure. The density of listings "
        "per neighbourhood, distance-price gradient, and room type mix all differ "
        "significantly, reflecting different regulatory environments, tourism "
        "patterns, and housing markets."
    ),
    implication=(
        "Cross-city investment strategies must account for local market dynamics. "
        "A pricing strategy that works in one city may not transfer directly "
        "to another."
    ),
    action=(
        "Investors evaluating multi-city Airbnb portfolios should weight "
        "their analysis by local regulatory risk, supply density, and "
        "the city-centre premium gradient unique to each market."
    ),
)))

In [ ]:
db.close()
print("\n✅ Notebook 02 complete — Geographic & Spatial Analysis")